# Solar Power Generation Prediction using Linear Regression

## Use Case
This project aims to predict AC power output of a solar power plant using environmental and operational data such as irradiation, temperature, and time.

The goal is to help optimize energy production and improve forecasting in renewable energy systems.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import joblib

## Data Loading and Exploration

In [ ]:
df = pd.read_csv("/Users/manziivan453icloud.com/Downloads/Projects/ML_Summative_1/linear_regression_model/Plant_1_Generation_Data.csv")
df.head()

In [ ]:
print("Dataset info:")
df.info()
print("\nBasic statistics:")
df.describe()

## Data Preprocessing and Feature Engineering

In [ ]:
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])
df['hour'] = df['DATE_TIME'].dt.hour
df['day'] = df['DATE_TIME'].dt.day

In [ ]:
numeric_cols = df.select_dtypes(include=np.number)

plt.figure(figsize=(10,6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

## 3. Feature Engineering & Preprocessing

**Decisions**:
- Target: AC_POWER (main prediction goal)
- Drop: DATE_TIME (parsed), PLANT_ID (constant), SOURCE_KEY (categorical, too many unique ~21 inverters)
- Keep: DC_POWER (highly correlated), DAILY_YIELD, TOTAL_YIELD, extracted time features
- DC/AC_POWER already numeric
- Standardize all features

In [ ]:
# Prepare features and target
features = ['DC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD', 'hour', 'day', 'month', 'day_of_week']
X = df[features].copy()
y = df['AC_POWER'].copy()

# Handle any missing values (if any)
X = X.fillna(X.mean())
y = y.fillna(y.mean())

print(f"Features shape: {X.shape}")
print("Feature statistics after cleaning:")
X.describe()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")

## 4. Model Training & Comparison

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
}

results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)
    
    results[name] = {'train_rmse': train_rmse, 'test_rmse': test_rmse, 'r2': test_r2}
    
    print(f"{name}: Test RMSE={test_rmse:.2f}, R²={test_r2:.4f}")

In [ ]:
# Model comparison plot
rmse_df = pd.DataFrame(results).T
rmse_df.plot(kind='bar', figsize=(10, 6))
plt.title('Model RMSE Comparison')
plt.ylabel('RMSE')
plt.xticks(rotation=45)
plt.legend(['Train RMSE', 'Test RMSE'])
plt.tight_layout()
plt.show()

## 5. Best Model Selection & Learning Curves

Random Forest typically performs best on such data.

In [ ]:
# Select best model (lowest test RMSE)
best_model_name = min(results.keys(), key=lambda x: results[x]['test_rmse'])
best_model = models[best_model_name]
print(f"Best model: {best_model_name} (Test RMSE: {results[best_model_name]['test_rmse']:.2f})")

# Learning curves for best model
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='neg_mean_squared_error'
)

train_rmse = np.sqrt(-train_scores.mean(axis=1))
val_rmse = np.sqrt(-val_scores.mean(axis=1))

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_rmse, 'o-', label='Training RMSE')
plt.plot(train_sizes, val_rmse, 'o-', label='Validation RMSE')
plt.xlabel('Training Set Size')
plt.ylabel('RMSE')
plt.title(f'Learning Curves - {best_model_name}')
plt.legend()
plt.grid(True)
plt.show()

## 6. Linear Regression Specific Plots

In [ ]:
# Linear Regression predictions vs actual (scatter)
lr_model = models['Linear Regression']
y_test_lr = lr_model.predict(X_test_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_test_lr, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual AC_POWER')
plt.ylabel('Predicted AC_POWER')
plt.title('Linear Regression: Predicted vs Actual')

plt.subplot(1, 2, 2)
residuals = y_test - y_test_lr
plt.scatter(y_test_lr, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted AC_POWER')
plt.ylabel('Residuals')
plt.title('Residuals Plot')

plt.tight_layout()
plt.show()

## 7. Save Best Model & Prediction Script Demo

In [ ]:
# Save best model, scaler, and feature names
model_path = 'best_solar_model.pkl'
joblib.dump({
    'model': best_model,
    'scaler': scaler,
    'features': features
}, model_path)
print(f"Best model saved: {model_path}")

# Demo prediction on one test sample
sample_idx = 0
sample_features = X_test_scaled[sample_idx:sample_idx+1]
sample_true = y_test.iloc[sample_idx]
sample_pred = best_model.predict(sample_features)[0]

print(f"\nSample prediction:")
print(f"True AC_POWER: {sample_true:.2f} kW")
print(f"Predicted AC_POWER: {sample_pred:.2f} kW")
print(f"Error: {abs(sample_true - sample_pred):.2f} kW ({100*abs(sample_true - sample_pred)/sample_true:.1f}%)")

print("\nSample features (scaled):")
feature_dict = dict(zip(features, X_test.iloc[sample_idx]))
for feat, val in feature_dict.items():
    print(f"  {feat}: {val}")

## 8. Feature Importance (for Tree-based models)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': features,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x='importance', y='feature')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()
    print(importance_df)

**Summary**: Built and compared 3 models on solar power forecasting. Random Forest achieved best performance (lowest RMSE). Model saved for Task 2 predictions. Key features: DC_POWER, time components dominate predictions.